# Introduction to vector databases

## Vector search

### Similar words

We have already established that words are represented as high-dimensional vectors in the vector space known as embeddings. Words are transformed into embeddings by using embedding techniques, such as BERT. Also of note is that similar words tend to have vectors (embeddings) that are positioned nearby in the vector space.

Without the loss of generality, let's suppose we have a vector space in two dimensions, where words are represented as two-dimensional points. The point (2, 3) denotes the word sunny while the point (3, 4) denotes the word sunlit. These two words have similar meanings. Hence, their vectors are located nearby in the vector space. On the other hand, the point (-2, -3.5) represents the word dark. Due to having the opposite meaning of sunny, the vector representing the word dark is situated apart from the vectors sunny and sunlit.

<img src="https://i.ibb.co/1J84S8SX/untitled2.jpg" alt="untitled2" border="0">

Moreover, a query can be mapped into high-dimensional vectors, and a match can be found in the vector space. In vector query execution, we aim to discover similar vectors to identify the most relevant candidates for our search results. If the vector content is indexed, the type of index used guides the search for relevant matches, which may be either exhaustive or focused on near neighbours. Also of note is that the latter facilitates faster processing. Once we pinpoint our candidates, we score the results using similarity metrics that measure the strength of the match.

Some popular algorithms used in vector search include KNN (K-Nearest Neighbours) and ANN (Approximate Nearest Neighbours). Both KNN and ANN are based on distance measurements to find the nearest neighbours. KNN finds the exact nearest neighbours, while ANN finds the approximate nearest neighbours. ANN can significantly improve search efficiency when compared to KNN when processing vast amounts of data, especially when dealing with high-dimensional vectors.

In the scope of this notebook, we will explore the KNN approach.

## Vector databases

Vector databases are highly specialized systems expertly designed for the efficient storage, retrieval, and management of high-dimensional vectors. Unlike traditional databases that limit themselves to scalar values like integers, vector databases excel in processing tasks that incorporate complex data types such as text, images, and audio mapped to multi-dimensional vector spaces, adeptly capturing relationships and similarities between data points. With the implementation of advanced indexing techniques and powerful similarity search algorithms, vector databases enable effective querying of large-scale datasets. This makes them essential tools for applications in natural language processing. 

One example represents the vector database pgvector, which we will later use for natural language processing tasks to store text embeddings in Programming Assignment 2. 

In the scope of this notebook, we will conduct simple KNN searches to obtain relevant information. First, we will start with the basics.

### System initialization

In this tutorial we re-use PostgreSQL database Docker image we have presented in *Web crawling - basic tools* notebook. If you already have tested the showcase example and have not deleted the container *postgres-wier* with the database, you can start the Docker container as indicated below.



<img src="https://i.ibb.co/DgCxJWdQ/docker-demo.png" alt="docker-demo" border="0">



Otherwise, follow next steps. 

First, prepare a file *database.sql*. The script will create a table with two rows:

``` sql
CREATE SCHEMA IF NOT EXISTS showcase;

CREATE TABLE showcase.counters (
    counter_id integer  NOT NULL,
    value integer NOT NULL,
    CONSTRAINT pk_counters PRIMARY KEY ( counter_id )
 );

INSERT INTO showcase.counters VALUES (1,0), (2,0);
```

Go to an empty folder and save the script into a subfolder named *init-scripts*. Create another empty folder named *pgdata*.

We run docker container using the following command. The command will name the container *postgresql-wier*, set username and password, map database files to folder *./pgdata* and initialization scripts to *./init-scripts*, map port 5432 to host machine (i.e. localhost) and run image *pgvector:pg17* in a detached mode. 

``` 
docker run --name postgresql-wier \
    -e POSTGRES_PASSWORD=SecretPassword \
    -e POSTGRES_USER=user \
    -e POSTGRES_DB=wier \
    -v $PWD/pgdata:/var/lib/postgresql/data \
    -v $PWD/init-scripts:/docker-entrypoint-initdb.d \
    -p 5432:5432 \
    -d pgvector/pgvector:pg17
```

If you use Command Prompt on Windows, the equivalent of the above command is as follows:

``` 
docker run --name postgresql-wier ^
    -e POSTGRES_PASSWORD=SecretPassword ^
    -e POSTGRES_USER=user ^
    -e POSTGRES_DB=wier ^
    -v "%CD%\pgdata:/var/lib/postgresql/data" ^
    -v "%CD%\init-scripts:/docker-entrypoint-initdb.d" ^
    -p 5432:5432 ^
    -d pgvector/pgvector:pg17
```

To check container's logs, run `docker logs -f postgresql-wier`.


### Getting started

We will enable the pgvector extension and insert a some example sentences in the database.  

But first, do check that you use the correct image for your database, i.e., *pgvector/pgvector:pg17*.  

Also, install the necessary dependencies.

In [1]:
%pip install pgvector
%pip install sentence_transformers
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 1.0 MB/s  0:00:10 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 1.5 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 1.4 MB/s  0:00:02 eta 0:00:010m
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 1.6 MB/s  0:00:01 eta 0:00:010m
Using cached h11-0.16.0-py3

In the next step, we will enable the pgvector extension in your PostgreSQL database (do this once in each database where you want to use it) with the following SQL statement:
```sql
CREATE EXTENSION IF NOT EXISTS vector
```

In [1]:
from pgvector.psycopg import register_vector
import psycopg

#connect to db
conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')

#enable `vector` extension if not already enabled
conn.execute('CREATE EXTENSION IF NOT EXISTS vector')
register_vector(conn)

ModuleNotFoundError: No module named 'psycopg'

Now, we will create two new tables *showcase.vector_demo* and *showcase.vector_demo2*, for storing embeddings. Both tables have similar column definitions (except for embedding vector size):
- **id**: primary key (unique for each sentence)
- **sentence**: text (content) of the sentence
- **embedding**: vector representation of the sentence.  

In [3]:
#delete tables vector_demo and vector_demo2 from the db if they exist
conn.execute('DROP TABLE IF EXISTS showcase.vector_demo')
conn.execute('DROP TABLE IF EXISTS showcase.vector_demo2')

#create tables vector_demo and vector_demo2 with columns id, content, and embedding
conn.execute('CREATE TABLE showcase.vector_demo (id bigserial PRIMARY KEY, sentence text, embedding vector(384))')
conn.execute('CREATE TABLE showcase.vector_demo2 (id bigserial PRIMARY KEY, sentence text, embedding vector(768))')

conn.close()

Next, we will define a list of sentences and calculate their embeddings using two different models from the [SentenceTransformer](https://sbert.net/) library, i.e.
- [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [LaBSE](https://huggingface.co/sentence-transformers/LaBSE)

In [4]:
from sentence_transformers import SentenceTransformer

#define sentences to be stored in the database
sentences = [
    'The sun is shining',
    'The sun shines with great brightness',
    'The sun shines',
    'The sun provides light and energy',
    'The sun shines brightly in the clear sky',
    'The sun shines brightly, warming the earth and providing light',
    'The clouds are covering the sky'    
]

#load SentenceTransformer model and generate embeddings
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)

model2 = SentenceTransformer('sentence-transformers/LaBSE')
embeddings2 = model2.encode(sentences)

print('Size of embedding: ' + str(embeddings.shape))
print('Size of embedding1: ' + str(embeddings2.shape))

print('\n\n\nembedding:\n ')
print(embeddings)

print('\n\n\nembedding 2:\n ')
print(embeddings2)


c:\Users\melanija.vezocnik\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Size of embedding: (7, 384)
Size of embedding1: (7, 768)



embedding:
 
[[ 0.00054292  0.10255139  0.08367033 ... -0.00199533 -0.03272489
   0.06198488]
 [ 0.00316647  0.08327056  0.08528873 ...  0.03729222 -0.02626909
   0.00733489]
 [-0.03556009  0.09748872  0.0818428  ...  0.00813875 -0.01296235
   0.02360048]
 ...
 [ 0.03556754  0.07777561  0.09210058 ...  0.03149207 -0.04171377
   0.02862354]
 [-0.02141308  0.07855915  0.11534784 ...  0.0253386   0.00757783
  -0.0016059 ]
 [-0.00135664  0.05125058  0.0965555  ... -0.06021433 -0.04138939
   0.1104074 ]]



embedding 2:
 
[[-0.07028948 -0.04968325 -0.03011552 ...  0.0662761  -0.00931124
  -0.01724292]
 [-0.05052446 -0.04258747 -0.02182684 ...  0.01908544  0.0320991
  -0.06173192]
 [-0.072459   -0.05111319 -0.02363922 ...  0.06433154 -0.00558642
  -0.01772116]
 ...
 [-0.05991964 -0.03267765 -0.0148006  ... -0.00967498 -0.00826744
  -0.05519555]
 [-0.06017009 -0.04150239 -0.05295153 ...  0.02691784 -0.01832757
  -0.02287312]
 [-0.071

After, we will store sentences in the database.

In [5]:
#print values in stored in the table showcase.vector_demo2
def print_db_values():

    """
    Print sentences and corresponding embeddings in the table showcase.vector_demo2.
    """
    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')

    retVal = []
    print("\nValues in the vector_demo2 table:")
    cur = conn.cursor()
    cur.execute("SELECT id, sentence, embedding FROM showcase.vector_demo2 ORDER BY id")
    for id, sentence, embedding in cur.fetchall():
        print(f"\Id: {id},   Sentence: {sentence},   Embedding: {embedding}")
        retVal.append({id: (sentence, embedding)})
    cur.close()
    conn.close()

    #return retVal

#insert a list of sentences and corresponding embeddings in the table showcase.vector_demo
def insert_db_sentences(sentences, embeddings):
    """
    Insert a list of sentences and corresponding embeddings in the table showcase.vector_demo.

    Parameters
    - sentences: A list of sentences to be inserted in the vector_demo table.
    - embeddings:  Embeddings to be inserted in the vector_demo table.
    """
    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 
    for sentence, embedding in zip(sentences, embeddings):
        embedding = embedding.tolist() #convert numpy array to python lists for compatibility with PostgreSQL
        cur.execute('INSERT INTO showcase.vector_demo (sentence, embedding) VALUES (%s, %s)', (sentence, embedding))
    cur.close()
    conn.close()


#insert a list of sentences and corresponding embeddings in the table showcase.vector_demo2
def insert_db_sentences2(sentences, embeddings):
    """
    Insert a list of sentences and corresponding embeddings in the table showcase.vector_demo2.

    Parameters
    - sentences: A list of sentences to be inserted in the vector_demo2 table.
    - embeddings:  Embeddings to be inserted in the vector_demo2 table.
    """
    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 
    for sentence, embedding in zip(sentences, embeddings):
        embedding = embedding.tolist() #convert numpy array to python lists for compatibility with PostgreSQL
        cur.execute('INSERT INTO showcase.vector_demo2 (sentence, embedding) VALUES (%s, %s)', (sentence, embedding))
    cur.close()
    conn.close()


insert_db_sentences(sentences, embeddings)
insert_db_sentences2(sentences, embeddings2)
print_db_values()



Values in the vector_demo2 table:
\Id: 1,   Sentence: The sun is shining,   Embedding: [-0.07028948,-0.049683254,-0.03011552,-0.012938374,0.05607148,-0.029962307,0.059932686,0.011225201,-0.075298995,0.0016879227,0.0009554222,0.003647062,0.047699496,0.025835184,0.047961555,-0.0030152267,-0.053688508,0.009652318,-0.026948724,-0.0025673392,0.00019981555,0.05638045,-0.0041230344,-0.045842383,-0.070369534,-0.036312956,-0.026046611,-0.009601733,-0.03641934,-0.039834738,-0.054666426,-0.004371423,0.031269927,-0.010603314,0.031432044,-0.07137101,-0.023846736,0.0012037104,-0.0052674534,0.005940936,-0.016637085,0.03484948,-0.024799095,0.043630254,0.0068362723,-0.030391425,-0.016110074,0.05867412,-0.06725076,0.020249117,-0.03913439,-0.018561509,-0.017509054,0.004243047,0.011734756,0.0020866015,0.02313671,-0.066070355,0.020998191,0.03245033,0.006295379,-0.047059774,0.013564305,-0.023117037,0.027465241,-0.052478783,-0.016250495,-0.04164084,-0.01427338,-0.007525287,-0.04052791,-0.01066069,-0.0298937

### Distance functions supported in pgvector database

Pgvector supports the following distance functions are:

- L2 distance (**`<->`**)
- (Negative) inner product (**`<#>`**)
- Cosine distance (**`<=>`**)
- L1 distance(**`<+>`**)
- Hamming distance (**`<~>`**): used for binary vectors
- Jaccard distance (**`<%>`**): used for binary vectors

####  L2 distance (`<->`)  

Let vectors $x$, $y \in \mathbb{R}^n$. L2 distance (the Euclidean distance) $d_{L2}(x, y)$ is defined as
$d_{L2}(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$.

#### (Negative) inner product (`<#>`)
The negative inner product of the dot product between two vectors. The inner product measures similarity, so negating it turns it into a pseudo-distance measure. For vectors $x$, $y \in \mathbb{R}^n$, the negative inner product $d_{\text{inner}}(x, y)$ is defined as
$d_{\text{inner}}(x, y) = - (x \cdot y)$.

#### Cosine distance (`<=>`)  
Cosine distance measures the angle between two vectors in a high-dimensional space. Let vectors $x$, $y \in \mathbb{R}^n$. The cosine distance is defined as
$d_{\cos}(x, y) = 1 - \frac{x \cdot y}{ \lVert x \rVert \lVert y \rVert}$
For cosine similarity, use $1 - d_{\cos}(x, y)$.

####  L1 Distance (`<+>`)  
L1 distance is also known as Manhattan distance. Let vectors $x$, $y \in \mathbb{R}^n$. The Manhattan distance is defined as:
$d_{L1}(x,y) = \sum_{i=1}^{n} |x_i - y_i|$

#### Hamming Distance (`<~>`)  
Hamming distance is used for binary vectors, as it counts the number of positions at which the corresponding elements differ. For vectors $x$, $y \in \{0, 1\}^n$, the Hamming distance is defined as 
$d_{Hamming}(x,y) = \sum_{i=1}^{n} |x_i - y_i|$. Obviously, $x_i - y_i \in \{0, 1\}$ for all $i \in {1, ..., n}$.
  

#### Jaccard Distance (`<%>`)  
Jaccard distance is used for binary vectors. It is derived from the Jaccard similarity. For vectors $x$, $y \in \{0, 1\}^n$, the Jaccard distance is defined as:
$d_{Jaccard}(x,y) = 1 - \frac{|x \cap y |}{| x \cup y |}$, where $ | x \cap y | $ is the number of positions where both vectors have 1s, and $ | x \cup y | $ is the number of positions where at least one vector has a 1.


### Querying pgvector database

Now that we have listed all the distance functions supported by the PNG vector database, we can showcase the differences in results using the above metrics. First, we will define functions for querying the database using the above distances for non-binary vectors.

In [6]:
#query using L2 distance
def query_db_L2(query, model_name, table_name):
    """
    The query_db_L2 function retrieves the top 5 most similar sentences from a pgvector database based on L2 (Euclidean) distance. 
    It uses a pre-trained SentenceTransformer model to encode the input query and then searches for the closest embeddings stored in the database.

    Parameters
    - query (str): The input text query to be searched.
    - model_name (str): The name of the SentenceTransformer model to be used for encoding the query.
    - table_name (str): The name of the table containing the stored sentence embeddings. Possible options are showcase.vector_demo and showcase.vector_demo2
    """
    
    #download the model
    model = SentenceTransformer(model_name)

    #calculate embedding for the query
    query_embedding = model.encode(query).tolist()  

    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 

    # execute the query to fetch the top 5 most similar sentences based on L2 distance
    result = cur.execute(
        'SELECT sentence, (embedding <-> %s::vector) AS distance '
        'FROM ' + table_name + ' '
        'ORDER BY embedding <-> %s::vector '
        'LIMIT 5',
        (query_embedding, query_embedding)  # pass the embedding twice, once for ordering and once for calculation
    ).fetchall()
    cur.close()
    conn.close()
    return result

#query using L1 distance
def query_db_L1(query, model_name, table_name):
    """
    The query_db_L1 function retrieves the top 5 most similar sentences from a pgvector database based on L1 (Manhattan) distance. 
    It uses a pre-trained SentenceTransformer model to encode the input query and then searches for the closest embeddings stored in the database.

    Parameters
    - query (str): The input text query to be searched.
    - model_name (str): The name of the SentenceTransformer model to be used for encoding the query.
    - table_name (str): The name of the table containing the stored sentence embeddings. Possible options are showcase.vector_demo and showcase.vector_demo2
    """
    
    #download the model
    model = SentenceTransformer(model_name)

    #calculate embedding for the query
    query_embedding = model.encode(query).tolist()  

    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 

    # execute the query to fetch the top 5 most similar sentences based on L1 distance
    result = cur.execute(
        'SELECT sentence, (embedding <+> %s::vector) AS distance '
        'FROM ' + table_name + ' '
        'ORDER BY embedding <+> %s::vector '
        'LIMIT 5',
        (query_embedding, query_embedding)  # pass the embedding twice, once for ordering and once for calculation
    ).fetchall()
    cur.close()
    conn.close()
    return result


#query using cosine distance
def query_db_cosine(query, model_name, table_name):
    """
    The query_db_cosine function retrieves the top 5 most similar sentences from a pgvector database based on cosine distance. 
    It uses a pre-trained SentenceTransformer model to encode the input query and then searches for the closest embeddings stored in the database.

    Parameters
    - query (str): The input text query to be searched.
    - model_name (str): The name of the SentenceTransformer model to be used for encoding the query.
    - table_name (str): The name of the table containing the stored sentence embeddings. Possible options are showcase.vector_demo and showcase.vector_demo2
    """
    
    #download the model
    model = SentenceTransformer(model_name)

    #calculate embedding for the query
    query_embedding = model.encode(query).tolist()  

    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 

    # execute the query to fetch the top 5 most similar sentences based on cosine distance
    result = cur.execute(
        'SELECT sentence, embedding <=> %s::vector AS distance '
        'FROM ' + table_name + ' '
        'ORDER BY embedding <=> %s::vector '
        'LIMIT 5',
        (query_embedding, query_embedding)  # pass the embedding twice, once for ordering and once for calculation
    ).fetchall()
    cur.close()
    conn.close()
    return result

#query using negative inner product
def query_db_inner(query, model_name, table_name):
    """
    The query_db_inner function retrieves the top 5 most similar sentences from a pgvector database based on (negative) inner product. 
    It uses a pre-trained SentenceTransformer model to encode the input query and then searches for the closest embeddings stored in the database.

    Parameters
    - query (str): The input text query to be searched.
    - model_name (str): The name of the SentenceTransformer model to be used for encoding the query.
    - table_name (str): The name of the table containing the stored sentence embeddings. Possible options are showcase.vector_demo and showcase.vector_demo2
    """
    
    #download the model
    model = SentenceTransformer(model_name)

    #calculate embedding for the query
    query_embedding = model.encode(query).tolist()  

    conn = psycopg.connect(host="localhost", dbname='wier', autocommit=True, password='SecretPassword', user='user')
    cur = conn.cursor() 

    # execute the query to fetch the top 5 most similar sentences based negative inner product
    result = cur.execute(
        'SELECT sentence, -(embedding <#> %s::vector) AS distance '
        'FROM ' + table_name + ' '
        'ORDER BY embedding <#> %s::vector '
        'LIMIT 5',
        (query_embedding, query_embedding)  # pass the embedding twice, once for ordering and once for calculation
    ).fetchall()
    cur.close()
    conn.close()
    return result


#print results
def print_results(result):
    """
    This function displays the results.

    Parameters
    - result: a list of tuples including sentence and embedding values
    """
    for i,(sentence, distance) in enumerate(result, start=1):
        print(f"{i}. {sentence} {distance}")

##### 1. Identical sentence (we query the same vector)

The query sentence "The sun is shining" is already stored in our database. For this reason, we obtain the perfect match across all distances.

In [7]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
query = 'The sun is shining'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun is shining 4.383406504629882e-07
2. The sun shines 0.3352121684414338
3. The sun shines brightly in the clear sky 0.6198005408498565
4. The sun shines with great brightness 0.6238624233689496
5. The sun provides light and energy 0.7193700355073457

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun is shining 6.851056241430342e-06
2. The sun shines 5.171389102935791
3. The sun shines with great brightness 9.485048294067383
4. The sun shines brightly in the clear sky 9.692066192626953
5. The sun provides light and energy 11.359393119812012

**Top 5 similar sentences using cosine distance:**

1. The sun is shining 0.0
2. The sun shines 0.05618351697921753
3. The sun shines brightly in the clear sky 0.19207626581192017
4. The sun shines with great brightness 0.19460210784083887
5. The sun provides light and energy 0.25874657981077176

**Top 5 similar sentences using negative inner product:**

1. The sun is shi

##### 2. Synonymous sentence (different wording, same meaning)

In this example, the query sentence is "The sun is shining brightly". We do not have the exact match stored in the database. Even though sentences have different wordings, they express the same idea, allowing us to see how cosine distance and inner product treat them as similar. In contrast, L2 and L1 distances show slight differences. 

In [8]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
query = 'The sun is shining brightly'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)



**Top 5 similar sentences using L2 distance:**

1. The sun is shining 0.29269569387783984
2. The sun shines 0.43540658102568464
3. The sun shines brightly in the clear sky 0.5268487090206557
4. The sun shines with great brightness 0.5505066856892993
5. The sun shines brightly, warming the earth and providing light 0.7208604925107082

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun is shining 4.488513946533203
2. The sun shines 6.681426525115967
3. The sun shines brightly in the clear sky 8.238308906555176
4. The sun shines with great brightness 8.43595027923584
5. The sun shines brightly, warming the earth and providing light 11.371321678161621

**Top 5 similar sentences using cosine distance:**

1. The sun is shining 0.042835354804992676
2. The sun shines 0.09478950500488281
3. The sun shines brightly in the clear sky 0.1387847661972046
4. The sun shines with great brightness 0.15152881001021856
5. The sun shines brightly, warming the earth and providing light 

#### 3. Contrasting sentence (antonyms)

If our query is "The moon is glowing", one can see that this sentence and sentences in the database are semantically opposite (e.g., sun versus moon, shining versus glowing). For this reason, the distance metrics show a higher degree of dissimilarity for them, particularly L2 and L1 distances.

In [9]:
query = 'The moon is glowing'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun shines with great brightness 0.9804734613677735
2. The sun shines 1.0057198733284474
3. The sun shines brightly in the clear sky 1.0088080287158576
4. The sun is shining 1.0126016659969495
5. The sun shines brightly, warming the earth and providing light 1.0133593009352868

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun shines with great brightness 15.200376510620117
2. The sun shines brightly in the clear sky 15.423728942871094
3. The sun is shining 15.66834831237793
4. The sun shines 15.676694869995117
5. The sun shines brightly, warming the earth and providing light 15.8329439163208

**Top 5 similar sentences using cosine distance:**

1. The sun shines with great brightness 0.48066411854815816
2. The sun shines 0.505736231803894
3. The sun shines brightly in the clear sky 0.5088467597961426
4. The sun is shining 0.5126810967922211
5. The sun shines brightly, warming the earth and providing light 0.51

#### 4. Short versus long sentences (different sentence lenghts)

The short sentences compared to the long ones will help see how cosine distance and inner product are slightly affected by the amount of content. However, overall, we still obtain similar results across all distances.

In [10]:
query = 'The sun shines in the sky, a gentle breeze rustles the leaves and birds chirp in harmony, welcoming a new day filled with endless possibilities'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun shines brightly in the clear sky 0.894329431413927
2. The sun shines 0.9140071363066162
3. The sun shines brightly, warming the earth and providing light 0.9337543365398243
4. The sun is shining 0.9420228577045603
5. The sun shines with great brightness 0.9812991816350218

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun shines brightly in the clear sky 13.75086784362793
2. The sun shines 14.561725616455078
3. The sun shines brightly, warming the earth and providing light 14.625229835510254
4. The sun is shining 15.069709777832031
5. The sun shines with great brightness 15.268866539001465

**Top 5 similar sentences using cosine distance:**

1. The sun shines brightly in the clear sky 0.39991259574890137
2. The sun shines 0.41770458221435547
3. The sun shines brightly, warming the earth and providing light 0.43594855070114136
4. The sun is shining 0.4437035322189331
5. The sun shines with great brightness 

#### 5. Different topics

Our query sentence includes an entirely different topic. As a consequence, the obtained results indicate a higher degree of dissimilarity.

In [11]:
query = 'Albert Einstein'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun shines 1.2448497051078857
2. The sun provides light and energy 1.274196936574429
3. The sun is shining 1.2832967181163561
4. The sun shines brightly, warming the earth and providing light 1.290931872076484
5. The sun shines with great brightness 1.2939578879842921

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun shines 19.350812911987305
2. The sun is shining 19.76311492919922
3. The sun provides light and energy 19.813377380371094
4. The sun shines with great brightness 20.05284881591797
5. The sun shines brightly, warming the earth and providing light 20.265907287597656

**Top 5 similar sentences using cosine distance:**

1. The sun shines 0.7748254239559174
2. The sun provides light and energy 0.8117890096777084
3. The sun is shining 0.8234252482652664
4. The sun shines brightly, warming the earth and providing light 0.8332525193691254
5. The sun shines with great brightness 0.8371635030854792

**Top 

#### 6. Minor changes (similar meaning but slight rewording)

The query "Shining is the sun" has a similar meaning as the sentence "The sun is shining", which is already included in the database. 

In [12]:
query = 'Shining is the sun'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun is shining 0.5517308321407458
2. The sun shines 0.6040598073268816
3. The sun shines with great brightness 0.7233008731496596
4. The sun shines brightly in the clear sky 0.7529179711920776
5. The sun shines brightly, warming the earth and providing light 0.8387384993195338

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun is shining 8.354349136352539
2. The sun shines 9.23680305480957
3. The sun shines with great brightness 11.259532928466797
4. The sun shines brightly in the clear sky 11.841655731201172
5. The sun shines brightly, warming the earth and providing light 13.233530044555664

**Top 5 similar sentences using cosine distance:**

1. The sun is shining 0.15220343159416572
2. The sun shines 0.18244408473705598
3. The sun shines with great brightness 0.2615820389514525
4. The sun shines brightly in the clear sky 0.2834427187774933
5. The sun shines brightly, warming the earth and providing light 0.

#### 7.  What about Slovene?

So far, we have tested several examples in English. Let's try one example sentence in Slovene, e.g. "Sonce sije svetlo na jasnem nebu". Its translation into English is "The sun is shining brightly in the clear sky". We already have this sentence stored in the database. 

`How will a query in Slovene impact the results?`

In [13]:
query = 'Sonce sije svetlo na jasnem nebu' #The sun is shining brightly in the clear sky
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
table_name = 'showcase.vector_demo'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun shines 1.3095693111771434
2. The sun provides light and energy 1.338079597882415
3. The sun is shining 1.3411985615361637
4. The sun shines with great brightness 1.350537699888187
5. The sun shines brightly in the clear sky 1.3682407354943475

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun shines 20.37506675720215
2. The sun provides light and energy 20.66716766357422
3. The sun is shining 20.834169387817383
4. The sun shines with great brightness 21.291563034057617
5. The sun shines brightly in the clear sky 21.412731170654297

**Top 5 similar sentences using cosine distance:**

1. The sun shines 0.8574858839818306
2. The sun provides light and energy 0.8952285721898071
3. The sun is shining 0.8994067147727777
4. The sun shines with great brightness 0.9119759973294732
5. The sun shines brightly in the clear sky 0.9360413812970291

**Top 5 similar sentences using negative inner product:**

1. The sun sh

We can see that the *all-MiniLM-L6-v2* model yields results that are not as expected due to the sentence "The sun shines" being the best option for all distances.

Some models, like the *LaBSE* model, are multilingual. Such models support different languages. Specifically, the *LaBSE* model supports both Slovene and English. Let's see how this model will perform.

In [14]:
query = 'Sonce sije svetlo na jasnem nebu' #The sun is shining brightly in the clear sky
model_name = 'sentence-transformers/LaBSE'
table_name = 'showcase.vector_demo2'

#L2 distance
resultL2 = query_db_L2(query, model_name, table_name)
print("\n**Top 5 similar sentences using L2 distance:**\n")
print_results(resultL2)

#L1 distance
resultL1 = query_db_L1(query, model_name, table_name)
print("\n**Top 5 similar sentences using L1 (Manhattan) distance:**\n")
print_results(resultL1)

#cosine distance
resultC = query_db_cosine(query, model_name, table_name)
print("\n**Top 5 similar sentences using cosine distance:**\n")
print_results(resultC)

#negative inner product
result_inner_product = query_db_inner(query, model_name, table_name)
print("\n**Top 5 similar sentences using negative inner product:**\n")
print_results(result_inner_product)


**Top 5 similar sentences using L2 distance:**

1. The sun shines brightly in the clear sky 0.3955300784024728
2. The sun shines with great brightness 0.7332246577048414
3. The sun provides light and energy 0.8502489861289095
4. The sun shines brightly, warming the earth and providing light 0.8998454464556783
5. The sun shines 0.9046301999830171

**Top 5 similar sentences using L1 (Manhattan) distance:**

1. The sun shines brightly in the clear sky 8.511971473693848
2. The sun shines with great brightness 15.930570602416992
3. The sun provides light and energy 17.948883056640625
4. The sun shines brightly, warming the earth and providing light 19.81377410888672
5. The sun shines 19.820674896240234

**Top 5 similar sentences using cosine distance:**

1. The sun shines brightly in the clear sky 0.07822206383281383
2. The sun shines with great brightness 0.2688091237062725
3. The sun provides light and energy 0.3614616559193544
4. The sun shines brightly, warming the earth and providing 

This model yields the expected results. 

In this tutorial, we have demonstrated that the model used for calculating embeddings plays a crucial role in vector retrieval. So far, we have utilized the KNN approach to search the database with non-binary vectors. 

You can test querying the database using Hamming and Jaccard distances for practice. To achieve this, you must select a model that calculates embeddings as binary vectors, create a new table for storing the embeddings and implement the code for querying using the Hamming and Jaccard distances. You can help with [this example](https://github.com/pgvector/pgvector-python/blob/master/examples/cohere/example.py).

## References

- [Understanding Vector databases.](https://learn.microsoft.com/en-us/data-engineering/playbook/solutions/vector-database/)
- [Book Vector Spaces First: Introduction to Linear Algebra](https://ruor.uottawa.ca/items/f66a4ede-e276-486c-9067-9621d5347440)
- [Vector database pgvector.](https://github.com/pgvector/pgvector)
- [pgvector-python](https://github.com/pgvector/pgvector-python)
- [Sentence transformers.](https://sbert.net/)
- [Model all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [Model LaBSE](https://huggingface.co/sentence-transformers/LaBSE)